In [1]:
import sys
sys.path.append('../')  # subir 1 nivel desde notebooks a src
import json
from src.core.scraper_agent import ScraperAgent
from src.core.validator_agent import ValidatorAgent
from src.core.writer_agent import WriterAgent
from src.sources.inventory.app import Inventory
from src.config.settings import COUNTRY

db_loader = Inventory()
scraper = ScraperAgent()
validator = ValidatorAgent()
writer = WriterAgent()


In [2]:
df_data_base = db_loader.load_db_from_country_selected()

CO


In [3]:
df_data_base_test = df_data_base[df_data_base["code"] == "CO2961-hero-hunk-125-r"]
df_data_base_test

,date,code,brand,model,year,type,technical_specs,publication_url,publication_image_url
76,11/02/2026,CO2961-hero-hunk-125-r,Hero,Hunk 125 R,2026,Urbana,"[{'key': 'ignition', 'value': 'CDI (Ignición p...",https://www.galgo.com/co/motos/CO2961-hero-hun...,https://images.ctfassets.net/8zlbnewncp6f/58pe...


In [4]:
for index, row in df_data_base_test.iterrows():
    marca = row["brand"]
    modelo = row["model"]
    año = row["year"]
    tipo = row["type"]
    pais = {"CO": "Colombia", "MX": "Mexico", "CL": "Chile"}.get(COUNTRY)
    ficha_tecnica = row["technical_specs"]
    print(f"Marca: {marca}, Modelo: {modelo}, Año: {año}, País: {pais}")

Marca: Hero, Modelo: Hunk 125 R, Año: 2026, País: Colombia


In [5]:
ficha_tecnica

"[{'key': 'ignition', 'value': 'CDI (Ignición por Descarga Capacitiva)', 'type': 'string', 'group': 'engine'}, {'key': 'riding_position', 'value': 'Recta', 'type': 'string', 'group': 'general'}, {'key': 'displacement', 'value': 124.7, 'type': 'number', 'group': 'engine'}, {'key': 'bore_and_stroke', 'value': '52.4 x 57.8 mm', 'type': 'string', 'group': 'engine'}, {'key': 'power', 'value': 10.72, 'type': 'number', 'group': 'engine'}, {'key': 'torque', 'value': 10.4, 'type': 'number', 'group': 'engine'}, {'key': 'engine_type', 'value': '4 tiempos, SOHC, 2 válvulas', 'type': 'string', 'group': 'engine'}, {'key': 'start_system', 'value': 'Eléctrico y Pedal', 'type': 'string', 'group': 'engine'}, {'key': 'front_brakes', 'value': 'Disco', 'type': 'string', 'group': 'brakes'}, {'key': 'rear_brakes', 'value': 'Tambor', 'type': 'string', 'group': 'brakes'}, {'key': 'front_tire', 'value': '90/90-17 (Tubeless/Sin Cámara)', 'type': 'string', 'group': 'wheels_and_tires'}, {'key': 'rear_tire', 'value

## 1. Scrapear data

In [6]:
experiencias_extraidas = scraper.scrape(
    marca=marca,
    modelo=modelo,
    año=año,
    pais=pais
)

c:\Users\JTRUJILLO\Documents\Galgo\Scripts\Otros\deep_research_models\notebooks\..\src\core\gemini_processor.py:9: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.client.interactions.create(


## 2. Validar data extraída

In [7]:
resultado_validacion = validator.validate(
    experiencias=experiencias_extraidas.get("experiencias_usuarios"),
    ficha_tecnica=ficha_tecnica,
    marca=marca,
    modelo=modelo,
    año=año,
    pais=pais
)

In [9]:
# Obtener experiencias que requieren re-research
experiencias_re_research = resultado_validacion.get("experiencias_requieren_re_research", [])

# Lista para almacenar resultados del re-research
experiencias_verificadas = []
experiencias_excluidas = []

In [11]:
for exp_re_research in experiencias_re_research:
    flag = exp_re_research.get("flag")
    experiencia_completa = exp_re_research.get("experiencia_completa", {})

    print(f"\nRe-research para: {experiencia_completa.get('fuente', 'N/A')}")
    print(f"Flag: {flag}")

    # Ejecutar re-research
    resultado_re = validator.re_research(
        experiencia=experiencia_completa,
        ficha_tecnica=ficha_tecnica,
        marca=marca,
        modelo=modelo,
        año=año,
        pais=pais,
        flag=flag
    )

    # Actualizar experiencia con resultado del re-research
    experiencia_actualizada = validator.update_experience_after_re_research(
        experiencia_original=experiencia_completa,
        resultado_re_research=resultado_re
    )

    # Clasificar según resultado
    resultado = resultado_re.get("resultado", "")
    if resultado in ["INCLUIR", "INCLUIR_CON_NOTA"]:
        experiencias_verificadas.append(experiencia_actualizada)
    elif resultado in ["EXCLUIR", "EXCLUIR_POR_PRECAUCION"]:
        experiencias_excluidas.append({
            "experiencia": experiencia_actualizada,
            "razon": resultado_re.get("razon", "")
        })

    # print(f"Resultado: {resultado}")
    # print(f"Razón: {resultado_re.get('razon', 'N/A')}")


Re-research para: https://www.youtube.com/watch?v=Common_Problems_Hero_Hunk_125R
Flag: ESPECIFICACION_AMBIGUA

Re-research para: https://digimotos.co/hero-hunk-125
Flag: CONTRADICCION_FRENOS


In [12]:
experiencias_finales = {
    "experiencias_validadas": resultado_validacion.get("experiencias_validadas", []),
    "experiencias_verificadas_incluir": experiencias_verificadas,
    "experiencias_excluidas": (
        resultado_validacion.get("experiencias_excluidas_automatico", []) +
        experiencias_excluidas
    )
}

In [13]:
experiencias_finales

{'experiencias_validadas': [{'fuente': 'https://www.youtube.com/watch?v=HeroHunk125R_Owner_Review_1',
   'pais_confirmado': True,
   'version_correcta': True,
   'confidence': 95,
   'extractos': [{'categoria': 'sensaciones_frenado',
     'texto': 'El frenado de esta moto me ha respondido muy bien, me gusta porque al accionar el freno se activa inmediatamente el sistema IBS, es potente.'},
    {'categoria': 'consumo',
     'texto': 'Ahorra mucho combustible, es una moto que ahorra mucha gasolina.'},
    {'categoria': 'problemas',
     'texto': "Cuando uno mete segunda suena muy fuerte, eso suena como 'tracata'. Es un sonido fuerte en la caja."},
    {'categoria': 'comodidad',
     'texto': 'Ergonómicamente la posición de los brazos en el manubrio queda perfecto. Las defensas son buenas, no son desechables.'},
    {'categoria': 'problemas',
     'texto': 'Nos tocó sacar una cita para revisión y fue muy larga, la moto se quedó quieta 8 días.'}],
   'menciones_specs_tecnicas': [{'spec': '

### Exportar resultados

In [14]:
if pais == "Colombia":
    pais = "CO"
elif pais == "Mexico":
    pais = "MX"
elif pais == "Chile":
    pais = "CL"

In [15]:
import json
nombre_archivo = f'{pais}_{marca}_{modelo}-informationv2.json'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    json.dump(experiencias_finales, f, ensure_ascii=False, indent=2)

print(f"\nResultados guardados en '{nombre_archivo}.json'")


Resultados guardados en 'CO_Hero_Hunk 125 R-informationv2.json.json'


## 3. Escritor

In [16]:
if pais == "Colombia":
    pais = "CO"
elif pais == "Mexico":
    pais = "MX"
elif pais == "Chile":
    pais = "CL"

### Información recolectada

In [17]:
nombre_archivo = f'{pais}_{marca}_{modelo}-information.json'
with open(nombre_archivo, 'r', encoding='utf-8') as f:
    experiencias_finales = json.load(f)

In [18]:
# Obtener experiencias validadas y verificadas
experiencias_validadas = experiencias_finales.get("experiencias_validadas", [])
experiencias_verificadas = experiencias_finales.get("experiencias_verificadas_incluir", [])

### Generar base de conocimiento

In [19]:
# Preparar ficha técnica (mismo formato que para validator)
ficha_tecnica_dict = {
    "brand": marca,
    "model": modelo,
    "year": str(año),
    "country": pais,
    "technical_specs": ficha_tecnica  # String raw
}

# Generar KB
knowledge_base = writer.write(
    experiencias_validadas=experiencias_validadas,
    experiencias_verificadas=experiencias_verificadas,
    ficha_tecnica=ficha_tecnica_dict,
    marca=marca,
    modelo=modelo,
    año=año,
    pais=pais,
    tipo=tipo  # Ej: "Urbana" (opcional)
)

In [20]:
print(knowledge_base)

[SEGMENTO]

Tipo según BD: Urbana
Segmento identificado por usuarios: Naked Urbana de entrada / Sport económica

[SENTIMIENTO]

Percibida como una "moto grande" en cuerpo de 125cc; los usuarios valoran su estética robusta y presencia visual por encima de su rendimiento mecánico. Se siente como una compra racional que sacrifica velocidad final y tecnología de motor (inyección) a cambio de apariencia, simplicidad mecánica y sensación de solidez estructural.

[SENSACIONES]

Estabilidad
Destacada por el ancho de sus llantas (120 atrás), inusual en el segmento. Se siente aplomada en curvas y transmite seguridad en línea recta, reduciendo la sensación de fragilidad típica de las 125cc delgadas.

Vibraciones
Bajas en régimen medio. El motor se percibe suave en ciudad, aunque a muy altas revoluciones (cerca del corte) transmite cierto estrés mecánico más que vibración molesta.

Frenado (validado por país)
Muy positivo. Aunque la ficha técnica indica CBS (Sistema de Frenos Combinados) y no ABS,

In [22]:
nombre_archivo = f'{pais}-{marca}_{modelo}-knowledge_base.md'
with open(nombre_archivo, 'w', encoding='utf-8') as f:
    f.write(knowledge_base)